# Evaluation Metrics For Retrieval

Evaluation metrics have two categories: offline and online. Offline metrics predict the system's performance before deployment.

Ground truth refers to the true relevance label of every item in the dataset. If an item is relevant, it is positive (p); if it is irrelevant, it is negative (n).

If an item is returned in the Top-K results, it is a predicted positive ($\hat{p}$). If it is not returned, it is a predicted negative ($\hat{n}$).

**When the retrieval system returns an item for a query:**
- If the item is relevant, it is a True Positive (TP).
- If the item is irrelevant, it is a False Positive (FP).

**When the retrieval system does NOT return an item for a query:**
- If the item is relevant, it is a False Negative (FN).
- If the item is irrelevant, it is a True Negative (TN).

## Recall@K

Recall@K measures how many relevant items were returned in the Top-K out of all the relevant items that exist in the entire dataset.
Total relevant items = (TP + FN)

$$ Recall@K = \frac{TP}{TP + FN} $$

Let $N$ be the total number of items in the entire dataset. $K$ is bounded as $K \in \{1, \dots, N\}$.

**Pros:**
* Recall@K is highly interpretable.

**Cons:**
* **Deceptive at high K:** By increasing K closer to N, the system will eventually return a perfect score, which can be deceptive.
* **Order-Unaware:** It does not account for the position of the relevant result. A correct result at rank 1 yields the same score as a correct result at rank K.

In [1]:
def recall(ground_truth, top_k, k):
    ground_truth_set = set(ground_truth)
    # Apply cutoff K to the retrieved list
    top_k_set = set(top_k[:k])
    
    # intersection length = True Positive (TP)
    # len(ground_truth_set) = True Positive (TP) + False Negative (FN)
    result = round(len(ground_truth_set & top_k_set) / float(len(ground_truth_set)), 2)
    return result

# --- Test Data ---
N = 6
ground_truth = [8, 100, 7] # 3 relevant items exist
top_k = [102, 8, 23, 7, 5] # 5 items returned by system

print("Testing Recall@K:")
for k in range(1, N):
    print(f"Recall@{k} = {recall(ground_truth, top_k, k)}")

Testing Recall@K:
Recall@1 = 0.0
Recall@2 = 0.33
Recall@3 = 0.33
Recall@4 = 0.67
Recall@5 = 0.67


## Mean Reciprocal Rank (MRR)

MRR is an order-aware metric calculated based on multiple queries. It evaluates the position of the first relevant document.

$$ MRR = \frac{1}{Q} \sum_{q=1}^{Q} \frac{1}{rank_q} $$

Where:
* $Q$ is the total number of queries.
* $q$ is a specific query.
* $rank_q$ is the rank (position) of the *first* ground truth result for query $q$. If no relevant result is found in the Top-K, $\frac{1}{rank_q}$ is 0.

In [7]:
ground_truth = [ [8], [7], [8] ]  # True items for every Query 
predicted = [
    [102, 8, 23, 7, 5],   #Q1
    [3, 12, 7, 9, 1],     #Q2
    [8, 5, 6, 2, 1],     #Q3
]

def reciprocal_rank(gt, pred):
    for rank, item in enumerate(pred, start=1):
        if item in gt:
            return 1 / rank
    return 0  

mrr = sum(reciprocal_rank(ground_truth[i], predicted[i]) for i in range(len(ground_truth))) / len(ground_truth)
print(mrr)

0.611111111111111


**Pros:**
* Order-aware. Very useful for use cases where the rank of the first relevant answer is critical, such as chatbots or Q&A systems.

**Cons:** 
* It evaluates only the rank of the first relevant item, ignoring the rest of the retrieved list. For example, if a system returns 10 items where only the 1st item is relevant and the other 9 are irrelevant, the MRR is a perfect 1.0. The metric fails to penalize the system for returning noise (irrelevant items) and does not reward the system if multiple relevant items exist further down the list.
* Less interpretable compared to Recall@K.

## Mean Average Precision (MAP)

There are a few steps to calculating MAP@K:

**Precision@K** = $$\frac{TruePositives}{TruePositives + False Negatives} = \frac{TP}{TP + FN} $$ 

Precision@k is similar to recall@k but we now consider the both relevant and non-relevant results only from from the returned items.

Note that the denominator in precision@K always equals $ K $ 
Precision@K = $$ \frac{TP}{K}$$


Next step of MAP@K is Average Precision AP@K:

**AP@K** $$ = \frac{\sum_{k=1}^{K} (Precision@k * rel_k)}{\text{number of relevant results}} $$

To get the Mean Average Precision@K (MAP@K) score for all queries, we simply divide by the number of queries $Q$:

$$ MAP@K = \frac{1}{Q} \sum_{q=1}^{Q} AP@K_q $$

**Pros:** Order-awared. Ideal for use cases where we expect to return multiple relevant items.

**Cons:** rel_K s binary. It does not give the results like less or more relevant than others.

In [3]:
ground_truth = [
    [2, 4, 8, 3],
    [8, 6, 5, 7],
    [3, 8] 
]

Q = len(ground_truth)
predicted = [1, 2, 3, 4, 5, 6, 7, 8]
k = 8
ap = []

for q in range(Q):
    ap_num = 0 
    for x in range(k):
        ground_truth_set = set(ground_truth[q])
        pred_set = set(predicted[:x+1])
        pred_at_k = len(ground_truth_set & pred_set) / (x+1)

        #calculated rel_k
        if predicted[x] in ground_truth[q]:
            rel_k = 1
        else:
            rel_k = 0

        ap_num += pred_at_k * rel_k 

    ap_q = ap_num / len(ground_truth[q])
    print(f"AP@{k}_{q+1} = {round(ap_q,2)}")
    ap.append(ap_q)


map_at_k = sum(ap) / Q

# generate results
print(f"mAP@{k} = {round(map_at_k, 2)}")
    

AP@8_1 = 0.6
AP@8_2 = 0.37
AP@8_3 = 0.29
mAP@8 = 0.42


## Normalized Discounted Cumulative Gain (NDCG@K):

Order-aware metric that we can derive from a few simpler metrics.

 Starting with Cumulative Gain (CG@K) calculated like so:

 $$\sum_{k=1}^{K} rel_k$$

 0 is least relevant, higher value most relevant. 

### Rank-Aware and Normalized Metrics: DCG and NDCG

These metrics are used to handle rank awareness and mathematically penalize or reward different levels of relevance.

#### 1. DCG@K (Discounted Cumulative Gain)

It penalizes relevant results that appear lower in the ranking by using a logarithmic decay function, $\log_2(1+k)$. The further down the result, the larger the logarithmic denominator becomes, decreasing that rank's contribution to the total score.

$$ DCG@K = \sum_{k=1}^{K} \frac{rel_k}{\log_2(1+k)} $$

* $k$: Rank of the retrieved item (from 1 to K).
* $rel_k$: Relevance score of the document at rank $k$ (can be binary or graded).

**Cons:** The absolute value of the DCG@K score is meaningless on its own. Because its range depends on the $rel_k$ scale chosen for the dataset (and is not inherently bounded between 0 and 1), it is highly uninterpretable.

#### 2. IDCG@K (Ideal DCG)

The "perfect ranking" baseline used to normalize the DCG. It is calculated by reordering the retrieved items from the highest relevance score ($rel_k$) to the lowest, and performing the standard DCG calculation. This represents the theoretical maximum score a query can achieve.

#### 3. NDCG@K (Normalized DCG)

Solves the interpretability problem of DCG. It is calculated by dividing the actual DCG score produced by the system by the theoretical maximum, the IDCG score. This normalization bounds the metric strictly within the $[0, 1]$ range, where 1.0 indicates a perfect ranking.

$$ NDCG@K = \frac{DCG@K}{IDCG@K} $$

In [4]:
from math import log2 

relevance = [0, 7, 2, 4, 6, 1, 4, 3]
K = 8

dcg = 0 
idcg = 0
ideal_relevance = sorted(relevance, reverse=True)

for k in range(1, K+1):
    dcg += relevance[k-1] / log2(1 + k)          
    idcg += ideal_relevance[k-1] / log2(1 + k) 
    ndcg = dcg / idcg

print(f"dcg: {dcg}")
print(f"ideal_relevance: {ideal_relevance}")
print(f"idcg: {idcg}")
print(f"ndcg: {ndcg}")

dcg: 16.71459088297532
ideal_relevance: [7, 6, 4, 4, 3, 2, 1, 0]
idcg: 16.71459088297532
ndcg: 1.0


**Pros:** Optimizes for highly relevant documents, is order-aware, and is easily interpretable.

**Cons:** We can't compare relativity to other items.

Source: https://www.pinecone.io/learn/offline-evaluation/#Metrics-in-Information-Retrieval